In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import wandb
import os
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances_argmin_min


newenv/lib64/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
newenv/lib64/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` stat

In [3]:
# Constants
NC = 24
NIC = 3
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- Multi-GPU Device Setup ---
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for Data Parallelism.")
    device = torch.device("cuda:0") # Set primary device to cuda:0
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using single device: {device}")

CENTROID_PATH = './data/color_centroids.npy'
os.makedirs('./data/output_images', exist_ok=True)


Using single device: cpu


newenv/lib64/python3.13/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
# Add this at the beginning to debug CUDA issues
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    print("CUDA is not available. Running on CPU.")

PyTorch version: 2.5.1+cu121
CUDA available: False
CUDA version: 12.1
Number of GPUs: 1
CUDA is not available. Running on CPU.


In [4]:
# ============================================================================
# DATASET
# ============================================================================
class ColorizationDataset(Dataset):
    def __init__(self, cifar_dataset, centroids, is_train=True):
        self.cifar_dataset = cifar_dataset
        self.centroids = centroids.astype(np.float32)
        self.is_train = is_train
        self.to_tensor = transforms.ToTensor()

        if is_train:
            self.augment = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
            ])
        else:
            self.augment = None

    def __len__(self):
        return len(self.cifar_dataset)

    def __getitem__(self, idx):
            img_pil, _ = self.cifar_dataset[idx]

            # Augmentation
            if self.augment is not None:
                img_pil = self.augment(img_pil)

            # RGB input (Used as input and for label computation)
            img_rgb_input = self.to_tensor(img_pil) # [3, 32, 32]

            # RGB for label computation (Flattened and as a numpy array)
            img_rgb_label_comp = img_rgb_input.permute(1, 2, 0).reshape(-1, 3).numpy()
            img_rgb_label_comp = img_rgb_label_comp.astype(np.float32)

            # Get nearest centroid labels
            labels = pairwise_distances_argmin_min(img_rgb_label_comp, self.centroids, metric='euclidean')[0]
            label_map = torch.tensor(labels, dtype=torch.long).reshape(32, 32)

            return img_rgb_input, label_map 

def load_cifar_data(batch_size=64, centroids=None):
    """Load CIFAR-10 data."""
    # Load datasets
    full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
    test_raw = datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

    # Split train/val (90/10)
    train_size = int(0.9 * len(full_train))
    val_size = len(full_train) - train_size
    train_subset, val_subset = random_split(full_train, [train_size, val_size])

    # Create wrapper datasets
    train_dataset = ColorizationDataset(
        [full_train[i] for i in train_subset.indices],
        centroids,
        is_train=True
    )

    val_dataset = ColorizationDataset(
        [full_train[i] for i in val_subset.indices],
        centroids,
        is_train=False
    )

    test_dataset = ColorizationDataset(test_raw, centroids, is_train=False)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, test_dataset



In [5]:
# ============================================================================
# MODEL
# ============================================================================
class ColorizationCNN(nn.Module):
    def __init__(self, NIC=3, NF=32, NC=24, kernel_size=3, dropout=0.3):
        super(ColorizationCNN, self).__init__()

        padding = kernel_size // 2

        # Encoder
        self.enc1 = self._conv_block(NIC, NF, kernel_size, padding)
        self.enc2 = self._conv_block(NF, 2 * NF, kernel_size, padding)
        self.enc3 = self._conv_block(2 * NF, 4 * NF, kernel_size, padding)

        # Decoder
        self.dec1 = self._deconv_block(4 * NF, 2 * NF, dropout=dropout)
        self.dec2 = self._deconv_block(2 * NF, NF, dropout=dropout)
        self.dec3 = self._deconv_block(NF, NC, dropout=0.0, final=True)

        # Classifier
        self.classifier = nn.Conv2d(NC, NC, kernel_size=1)

        self._initialize_weights()

    def _conv_block(self, in_c, out_c, kernel_size, padding):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def _deconv_block(self, in_c, out_c, dropout=0.3, final=False):
        layers = [
            nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_c)
        ]
        if not final:
            layers.append(nn.ReLU(inplace=True))
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)
        x = self.classifier(x)
        return x



In [6]:
# ============================================================================
# TRAINING
# ============================================================================
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += pixel_accuracy(outputs, targets)

        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)}: Loss={loss.item():.4f}')

    return total_loss / len(loader), total_acc / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            total_acc += pixel_accuracy(outputs, targets)

    return total_loss / len(loader), total_acc / len(loader)


@torch.no_grad()
def pixel_accuracy(logits, targets):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == targets).sum().item()
    total = preds.numel()
    return correct / total


def compute_class_weights(train_loader, NC, device):
    """Compute balanced class weights from actual data."""
    print("\nComputing class weights from training data...")
    all_labels = []

    for batch_idx, (_, labels) in enumerate(train_loader):
        all_labels.append(labels.view(-1).cpu().numpy())
        if batch_idx >= 100:  # Sample first 100 batches for speed
            break

    all_labels = np.concatenate(all_labels)
    counts = np.bincount(all_labels, minlength=NC)

    print(f"Class distribution (first 100 batches):")
    for i in range(NC):
        if counts[i] > 0:
            pct = 100 * counts[i] / counts.sum()
            print(f"  Cluster {i:2d}: {counts[i]:7d} pixels ({pct:5.2f}%)")

    # Compute weights
    freq = counts / counts.sum()
    weights = 1.0 / (freq + 0.01)  # Smoothing
    weights = weights / weights.sum() * NC

    return torch.tensor(weights, dtype=torch.float32).to(device)


def logits_to_rgb(logits, centroids):
    """Convert logits to RGB using centroids."""
    device = logits.device
    B, NC, H, W = logits.shape
    preds = torch.argmax(logits, dim=1).view(-1)
    centroids_t = torch.from_numpy(centroids).to(device)
    rgb_flat = centroids_t[preds]
    return rgb_flat.view(B, H, W, 3).permute(0, 3, 1, 2)


def generate_colorized_images(model, test_dataset, centroids, device, num_images=10):
    """Generate and display colorized images."""
    model.eval()
    fig, axes = plt.subplots(3, num_images, figsize=(num_images * 2, 6))

    with torch.no_grad():
        for i in range(num_images):
            # The input is now RGB, not grayscale
            rgb_input, _ = test_dataset[i]
            original_rgb = transforms.ToTensor()(test_dataset.cifar_dataset[i][0])

            rgb_batch = rgb_input.unsqueeze(0).to(device)
            logits = model(rgb_batch)
            predicted_rgb = logits_to_rgb(logits.cpu(), centroids).squeeze(0)

            # Plot
            axes[0, i].imshow(original_rgb.permute(1, 2, 0))
            axes[0, i].axis('off')
            axes[1, i].imshow(rgb_input.permute(1, 2, 0)) # Show RGB input
            axes[1, i].axis('off')
            axes[2, i].imshow(predicted_rgb.permute(1, 2, 0).clamp(0, 1))
            axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=12)
    axes[1, 0].set_ylabel('RGB Input', fontsize=12) # <-- MODIFIED Label
    axes[2, 0].set_ylabel('Predicted Cluster RGB', fontsize=12) # <-- MODIFIED Label

    plt.tight_layout()
    plt.savefig('colorized_results.png', dpi=150, bbox_inches='tight')

    try:
        wandb.log({"colorized_images": wandb.Image(plt)})
    except:
        pass

    plt.show()

In [7]:

# ============================================================================
# MAIN
# ============================================================================
def main():
    # Load centroids
    centroids = np.load(CENTROID_PATH)
    if centroids.max() > 1.5:
        centroids = centroids / 255.0
    centroids = centroids.astype(np.float32)
    print(f"Loaded centroids: {centroids.shape}, range [{centroids.min():.4f}, {centroids.max():.4f}]")

    # Load data
    print("\nLoading CIFAR-10 data...")
    train_loader, val_loader, test_loader, test_dataset = load_cifar_data(
        batch_size=64,
        centroids=centroids
    )

    # Model
    model = ColorizationCNN(NIC=1, NF=32, NC=24, kernel_size=3, dropout=0.3)
    # --- DataParallel Wrapper ---
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        # Move the DataParallel wrapper to the primary device (cuda:0)
        model.to(device)
    else:
        model.to(device)
    # --------------------------

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Loss and optimizer
    class_weights = compute_class_weights(train_loader, NC, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    if isinstance(model, nn.DataParallel):
        params = model.module.parameters()
    else:
        params = model.parameters()

    optimizer = torch.optim.AdamW(params, lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # WandB
    wandb.init(project="cifar10-colorization", config={
        "learning_rate": 3e-4,
        "epochs": 25,
        "batch_size": 64,
        "NF": 32,
        "dropout": 0.3,
        "optimizer": "AdamW",
        "label_smoothing": 0.1
    })

    # Training loop
    best_val_loss = float("inf")
    patience_counter = 0
    PATIENCE = 7
    EPOCHS = 25

    for epoch in range(EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"{'='*60}")

        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]['lr']
        })

        # Save best model
        if val_loss < best_val_loss - 1e-4:
            print(f"✓ Validation improved: {best_val_loss:.4f} → {val_loss:.4f}")
            best_val_loss = val_loss
            patience_counter = 0
            state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save({
                'epoch': epoch,
                'model_state_dict': state_dict_to_save,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, "best_colorization_model_nic3.pt")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    # Load best and visualize
    checkpoint = torch.load(f"best_{wandb.run.id}_nic3.pt")
    # Load the state dict back into the UNWRAPPED model for visualization
    # Re-initialize a single-device model for loading
    final_model = ColorizationCNN(NIC=NIC, NF=32, NC=24, kernel_size=3, dropout=0.3).to(device)
    final_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\n✓ Loaded best model from epoch {checkpoint['epoch']+1}")

    generate_colorized_images(model, test_dataset, centroids, device, num_images=10)

    wandb.finish()
    print("\n✓ Training complete!")



In [8]:
def generate_colorization_figure(model, test_dataset, centroids, device, num_images=8):
    """
    Generate a figure showing original, RGB input, and predicted cluster RGB images.
    """
    model.eval()

    fig, axes = plt.subplots(3, num_images, figsize=(num_images * 2, 6))

    with torch.no_grad():
        for i in range(num_images):
            # Get RGB input and original RGB
            rgb_input, _ = test_dataset[i]
            original_rgb = transforms.ToTensor()(test_dataset.cifar_dataset[i][0])

            # Generate colorized prediction
            rgb_batch = rgb_input.unsqueeze(0).to(device)  # [1, 3, 32, 32]
            logits = model(rgb_batch)  # [1, 24, 32, 32]

            # Convert logits to RGB
            predicted_rgb = logits_to_rgb(logits, centroids)
            predicted_rgb = predicted_rgb.squeeze(0).cpu()  # [3, 32, 32]

            # Plot
            axes[0, i].imshow(original_rgb.permute(1, 2, 0))
            axes[0, i].axis('off')

            axes[1, i].imshow(rgb_input.permute(1, 2, 0)) # Show RGB input
            axes[1, i].axis('off')

            axes[2, i].imshow(predicted_rgb.permute(1, 2, 0).clamp(0, 1))
            axes[2, i].axis('off')

    # Add row labels
    axes[0, 0].set_ylabel('Original', fontsize=12, rotation=0, labelpad=40, va='center')
    axes[1, 0].set_ylabel('RGB Input', fontsize=12, rotation=0, labelpad=40, va='center') # <-- MODIFIED Label
    axes[2, 0].set_ylabel('Predicted Cluster RGB', fontsize=12, rotation=0, labelpad=40, va='center') # <-- MODIFIED Label

    plt.tight_layout()

    return fig

import wandb

In [ ]:
# ============================================================================
# STEP 1: Define Sweep Configuration
# ============================================================================
sweep_config = {
    "name": "cifar10-cluster-prediction-hyperparameter-sweep-nic3", # <-- MODIFIED name
    "method": "random",
    "metric": {
        "name": "best_val_loss",
        "goal": "minimize"
    },
    "parameters": {
        "learning_rate": {
            "values": [1e-4, 3e-4, 1e-3, 3e-3]
        },
        "batch_size": {
            "values": [32, 64, 128]
        },
        "num_filters": {
            "values": [8, 16, 32]
        },
        "kernel_size": {
            "values": [3, 5]
        },
        "optimizer": {
            "values": ["SGD", "Adam"]
        },
        "dropout": {
            "values": [0.2, 0.3, 0.4]
        }
    }
}

# ============================================================================
# STEP 2: Define Training Function for Sweep
# ============================================================================
def train_sweep_run(config=None):
    """
    Training function called by wandb agent for each sweep configuration.
    """
    with wandb.init(config=config):
        cfg = wandb.config

        # Load centroids
        centroids = np.load(CENTROID_PATH)
        if centroids.max() > 1.5:
            centroids = centroids / 255.0
        centroids = centroids.astype(np.float32)

        # Load data with sweep batch size
        train_loader, val_loader, test_loader, test_dataset = load_cifar_data(
            batch_size=cfg.batch_size,
            centroids=centroids
        )

        # Initialize model with sweep parameters (NIC is hardcoded as 3 in the class)
        model = ColorizationCNN(
            NIC=NIC, # Use NIC=3
            NF=cfg.num_filters,
            NC=24,
            kernel_size=cfg.kernel_size,
            dropout=cfg.dropout
        )

        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model) 
            model.to(device)
        else:
            model.to(device)

        # Compute class weights
        class_weights = compute_class_weights(train_loader, NC, device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        # NOTE: Access the unwrapped model's parameters if DataParallel is used
        if isinstance(model, nn.DataParallel):
            params = model.module.parameters()
        else:
            params = model.parameters()

        # Optimizer based on config
        if cfg.optimizer == "SGD":
            optimizer = torch.optim.SGD(
                params,
                lr=cfg.learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
        else:  # Adam
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=cfg.learning_rate,
                weight_decay=1e-4
            )

        # Scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )

        # Training loop
        best_val_loss = float("inf")
        patience_counter = 0
        PATIENCE = 3
        EPOCHS = 25

        for epoch in range(EPOCHS):
            # Train
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

            # Validate
            val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

            # Update scheduler
            scheduler.step(val_loss)

            # Log to wandb
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "learning_rate": optimizer.param_groups[0]['lr']
            })

            print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                # Save checkpoint
                torch.save({
                    'model_state_dict': state_dict_to_save,
                    'val_loss': val_loss,
                    'config': dict(cfg)
                }, f"best_{wandb.run.id}_nic3.pt") # <-- MODIFIED filename

            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

        # Log best validation loss
        wandb.log({"best_val_loss": best_val_loss})

        # Generate example images with best model
        checkpoint = torch.load(f"best_{wandb.run.id}_nic3.pt") # <-- MODIFIED filename
        # Initialize a single-device model for visualization
        final_model = ColorizationCNN(
            NIC=NIC,
            NF=cfg.num_filters,
            NC=24,
            kernel_size=cfg.kernel_size,
            dropout=cfg.dropout
        ).to(device)
        
        final_model.load_state_dict(checkpoint['model_state_dict']) 

        # Generate example colorizations with best model
        print("\nGenerating example images...")
        fig = generate_colorization_figure(final_model, test_dataset, centroids, device, num_images=8)

        # Log final results
        wandb.log({
            "best_val_loss": best_val_loss,
            "example_colorizations": wandb.Image(fig)
        })
        plt.close(fig)

        # Clean up
        # os.remove(f"best_{wandb.run.id}_nic3.pt") # <-- MODIFIED filename

        return best_val_loss


# ============================================================================
# STEP 3: Initialize and Run Sweep
# ============================================================================

# Initialize the sweep
sweep_id = wandb.sweep(
    sweep_config,
    project="cifar10-colorization-sweep-nic3" # <-- MODIFIED project name
)

print(f"Sweep ID: {sweep_id}")
print(f"Sweep URL will be available after first run starts")

# Run the sweep agent (20 configurations)
wandb.agent(sweep_id, function=train_sweep_run, count=20)

print(f"\n✓ Sweep completed!")
print(f"View results at: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep-nic3/sweeps/{sweep_id}")



In [16]:
def print_best_results_robust(current_sweep_id):
    """
    Analyzes sweep results by calculating the minimum val_loss from run history,
    bypassing the corrupted summary data.
    """
    api = wandb.Api()
    
    try:
        sweep = api.sweep(f"gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep-nic3/{current_sweep_id}")
    except Exception as e:
        print(f"\nError retrieving sweep. Check username and project name. Details: {e}")
        return None
    
    run_metrics = []
    print("\nAttempting robust data retrieval from run history...")

    # --- 1. Retrieve and Calculate Best Loss from History for Each Run ---
    for run in sweep.runs:
        try:
            # Fetch the entire history table, specifically the val_loss column
            history_df = run.history(keys=['val_loss'])
            
            if history_df.empty or 'val_loss' not in history_df.columns:
                min_loss = float("inf")
                print(f"Warning: Run {run.name} ({run.id}) has no 'val_loss' history.")
            else:
                # Find the minimum validation loss recorded throughout the run
                min_loss = history_df['val_loss'].min()
            
            # Safely get config - convert to dict if it's not already
            config = run.config
            if isinstance(config, str):
                import json
                try:
                    config = json.loads(config)
                except:
                    config = {}
            elif not isinstance(config, dict):
                config = dict(config) if hasattr(config, '__iter__') else {}
                
            run_metrics.append({
                'run': run,
                'min_val_loss': min_loss,
                'config': config
            })
            
        except Exception as e:
            # Catch errors during history fetching
            print(f"ERROR: Failed to retrieve history for run {run.name} ({run.id}). Skipping. Error: {e}")
            run_metrics.append({
                'run': run,
                'min_val_loss': float("inf"),
                'config': {}
            })

    # --- 2. Sort Runs by the Calculated Minimum Loss ---
    runs_sorted = sorted(run_metrics, key=lambda x: x['min_val_loss'])

    if not runs_sorted or runs_sorted[0]['min_val_loss'] == float("inf"):
        print("\n" + "="*70)
        print("COULD NOT DETERMINE BEST RUN: All runs failed or lack history data.")
        print("="*70)
        return None

    # --- 3. Print Results ---
    best_result = runs_sorted[0]
    best_run = best_result['run']
    
    print("\n" + "="*70)
    print("BEST HYPERPARAMETER CONFIGURATION (NIC=3) - HISTORY ANALYSIS")
    print("="*70)

    print(f"\nRun Name: {best_run.name}")
    print(f"Run ID: {best_run.id}")
    print(f"Run URL: {best_run.url}")
    print(f"\nCalculated Best Validation Loss: {best_result['min_val_loss']:.6f}")

    print(f"\nOptimal Hyperparameters:")
    config = best_result['config']
    if config:
        print(f"  Learning Rate:  {config.get('learning_rate', 'N/A')}")
        print(f"  Batch Size:     {config.get('batch_size', 'N/A')}")
        print(f"  Num Filters:    {config.get('num_filters', 'N/A')}")
        print(f"  Kernel Size:    {config.get('kernel_size', 'N/A')}")
        print(f"  Optimizer:      {config.get('optimizer', 'N/A')}")
        print(f"  Dropout:        {config.get('dropout', 'N/A')}")
    else:
        print("  No config data available")

    print(f"\n" + "="*70)
    print("TOP 5 CONFIGURATIONS")
    print("="*70)

    for i, result in enumerate(runs_sorted[:5], 1):
        loss_str = f"{result['min_val_loss']:.6f}" if result['min_val_loss'] != float("inf") else "FAILED/N/A"
        config = result['config']
        
        print(f"\n{i}. {result['run'].name}")
        print(f"   Val Loss: {loss_str}")
        if config:
            print(f"   LR={config.get('learning_rate', 'N/A')}, "
                  f"BS={config.get('batch_size', 'N/A')}, "
                  f"NF={config.get('num_filters', 'N/A')}, "
                  f"KS={config.get('kernel_size', 'N/A')}")
            print(f"   Optimizer={config.get('optimizer', 'N/A')}, "
                  f"Dropout={config.get('dropout', 'N/A')}")
        else:
            print("   No config available")
        print(f"   URL: {result['run'].url}")

    print("\n" + "="*70)

    return best_run

# Run the analysis
best_run = print_best_results_robust(sweep_id)


Attempting robust data retrieval from run history...

BEST HYPERPARAMETER CONFIGURATION (NIC=3) - HISTORY ANALYSIS

Run Name: dandy-sweep-1
Run ID: shcbt66w
Run URL: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep-nic3/runs/shcbt66w

Calculated Best Validation Loss: 1.757800

Optimal Hyperparameters:
  Learning Rate:  {'value': 0.001}
  Batch Size:     {'value': 64}
  Num Filters:    {'value': 32}
  Kernel Size:    {'value': 3}
  Optimizer:      {'value': 'Adam'}
  Dropout:        {'value': 0.3}

TOP 5 CONFIGURATIONS

1. dandy-sweep-1
   Val Loss: 1.757800
   LR={'value': 0.001}, BS={'value': 64}, NF={'value': 32}, KS={'value': 3}
   Optimizer={'value': 'Adam'}, Dropout={'value': 0.3}
   URL: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep-nic3/runs/shcbt66w

2. lemon-sweep-19
   Val Loss: 1.774259
   LR={'value': 0.0003}, BS={'value': 64}, NF={'value': 32}, KS={'value': 3}
   Optimizer={'value': 'Adam'}, Dropout={'value': 0.3}
   URL:

# WandB Links
https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep-nic3?nw=nwusergracygarg6516
